# ChatBot Workflow

In [1]:
from langgraph.graph import START , StateGraph , END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver

In [2]:
load_dotenv()
llmModel = ChatGroq(model='openai/gpt-oss-120b')

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = llmModel.invoke(messages)

    return {'messages':[response]}

In [ ]:
checkpoint = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpoint)

In [ ]:
chatbot

In [ ]:
thread_id = "1"

while True:
    user_message = input('Type here:')
    print('user:', user_message)

    if user_message.strip().lower() in ['exit', 'stop', 'bye']:
         break

    initial_state = {
    'messages':[HumanMessage(content=user_message)]
    }

    config = {'configurable' : {'thread_id': thread_id}}

    response = chatbot.invoke(initial_state, config=config)

    print('AI:',response['messages'][-1].content)

In [ ]:
thread_id = "2"

initial_state = {
    'messages':[HumanMessage(content='What is my name')]
}
config = {'configurable':{'thread_id':thread_id}}
response = chatbot.invoke(initial_state, config=config)

response['messages'][-1].content